# 🚀 Kaggle TPU-Friendly Fine-Tuning: Industrial Telemetry Edge LLM

This notebook fine-tunes **`HuggingFaceTB/SmolLM2-135M-Instruct`** (or `Qwen/Qwen2.5-0.5B-Instruct`) for the **Industrial Telemetry Chatbot**.

### Features:
- **Auto-detects Dataset**: Searches `/kaggle/input` automatically for your uploaded `train_chatml.jsonl` and `val_chatml.jsonl`.
- **TPU & GPU Hybrid Support**: Configured with PyTorch-XLA / bfloat16 optimizations for Kaggle TPU VM (v3-8 / v5e) and automatic fallback for GPU T4/P100.
- **Zero Dynamic-Shape Recompilation**: Uses `max_seq_length=512` and `dataloader_drop_last=True` for peak TPU execution speed.
- **1-Click Zip Output**: Merges LoRA weights and creates a downloadable `.zip` file in `/kaggle/working` for edge deployment.

In [ ]:
# 1. Install fine-tuning dependencies
!pip install -q transformers datasets peft trl accelerate

In [ ]:
# 2. Auto-Detect Dataset in /kaggle/input (from uploaded ZIP)
import os

def find_dataset_files():
    search_roots = ['/kaggle/input', '.', './data']
    train_candidates = []
    val_candidates = []
    
    for root in search_roots:
        if os.path.exists(root):
            for dirpath, _, filenames in os.walk(root):
                for f in filenames:
                    full_path = os.path.join(dirpath, f)
                    lower = f.lower()
                    if f.endswith('.jsonl') or f.endswith('.json'):
                        if 'train' in lower:
                            train_candidates.append(full_path)
                        elif 'val' in lower or 'test' in lower or 'eval' in lower:
                            val_candidates.append(full_path)
                            
    if not train_candidates or not val_candidates:
        # Fallback to any jsonl files found
        all_jsonl = [os.path.join(dp, f) for dp, _, fs in os.walk('/kaggle/input') for f in fs if f.endswith('.jsonl')]
        if len(all_jsonl) >= 2:
            train_candidates = [all_jsonl[0]]
            val_candidates = [all_jsonl[1]]
            
    if not train_candidates:
        raise FileNotFoundError("❌ Could not find train dataset in /kaggle/input! Please upload dataset_for_kaggle.zip into Kaggle Input.")
    if not val_candidates:
        raise FileNotFoundError("❌ Could not find validation dataset in /kaggle/input! Please upload dataset_for_kaggle.zip into Kaggle Input.")
        
    train_path = sorted(train_candidates)[0]
    val_path = sorted(val_candidates)[0]
    
    print(f"✅ Auto-detected Train dataset:      {train_path}")
    print(f"✅ Auto-detected Validation dataset: {val_path}")
    return train_path, val_path

TRAIN_FILE, VAL_FILE = find_dataset_files()

In [ ]:
# 3. Detect Accelerator (TPU vs GPU)
import torch

IS_TPU = False
try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    IS_TPU = True
    print(f"🚀 Active Accelerator: KAGGLE TPU ({device})")
except Exception:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"⚡ Active Accelerator: {device}")

In [ ]:
# 4. Load Base Model & Configure LoRA
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
# You can also swap to "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Loading tokenizer for: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if (IS_TPU or (torch.cuda.is_available() and torch.cuda.is_bf16_supported())) else (torch.float16 if torch.cuda.is_available() else torch.float32)

model_kwargs = {
    "torch_dtype": dtype,
    "trust_remote_code": True
}
if not IS_TPU and torch.cuda.is_available():
    model_kwargs["device_map"] = "auto"

print(f"Loading base model ({dtype})...")
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 5. Configure TPU-Optimized Training Arguments & Train
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

use_bf16 = IS_TPU or (torch.cuda.is_available() and torch.cuda.is_bf16_supported())
use_fp16 = (not use_bf16) and torch.cuda.is_available()
dataset = load_dataset("json", data_files={"train": TRAIN_FILE, "validation": VAL_FILE})
print(f"Loaded {len(dataset['train'])} training samples, {len(dataset['validation'])} validation samples.")

training_args = SFTConfig(
    output_dir="./training_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=16 if IS_TPU else 8,
    per_device_eval_batch_size=16 if IS_TPU else 8,
    learning_rate=2e-4,
    warmup_steps=100,
    max_length=512,  # SFTConfig uses max_length
    loss_type='nll',
    weight_decay=0.01,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    bf16=use_bf16,
    fp16=use_fp16,
    dataloader_drop_last=True if IS_TPU else False,  # Prevents TPU recompilation on uneven final batch
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=training_args,
)

print("🚀 Starting fine-tuning...")
trainer.train()

In [ ]:
# 6. Merge LoRA with Base Model for Direct Deployment
import os

OUTPUT_DIR = "/kaggle/working/smollm2-135m-industrial-merged"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Merging LoRA adapter back into base model: {OUTPUT_DIR}")
merged_model = model.merge_and_unload()
merged_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✅ Merged model successfully saved!")

In [ ]:
# 7. Verification Test with Industrial Telemetry Query
test_messages = [
    {
        "role": "system",
        "content": "You extract report intent for an offline industrial asset PDF bot. Return strict JSON only.\n\nASSETS: 1: Boiler Feed Pump Motor | 2: Air Compressor Motor | 3: Main Conveyor Drive Motor | 4: Cooling Tower Fan Motor | 5: Primary Crusher Motor\n\nValid task: choose an asset and timeframe for a telemetry PDF report.\nOff-topic: sports, weather, news, jokes, general chat, coding, personal questions.\n\nJSON:\n{\"isOnTopic\":bool,\"isComplete\":bool,\"replyMessage\":\"\",\"suggestedAction\":\"select_asset\"|\"select_timeframe\"|\"confirm_queue\"|\"none\",\"suggestedOptions\":[],\"extractedParameters\":{\"assetId\":int|null,\"assetName\":\"str\"|null,\"timeRange\":\"str\"|null,\"fromDate\":\"str\"|null,\"toDate\":\"str\"|null,\"mode\":\"raw\"}}"
    },
    {
        "role": "user",
        "content": "CURRENT_STATE: asset=null,time=null\nINPUT: \"crusher motor for past 2 weeks\""
    }
]

prompt_text = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt").to(merged_model.device)

with torch.no_grad():
    outputs = merged_model.generate(**inputs, max_new_tokens=100, temperature=0.1, do_sample=False)

raw_response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- Verification Output ---")
print(raw_response)

In [ ]:
# 8. Package Merged Model into a Downloadable ZIP
import shutil

zip_name = "/kaggle/working/smollm2_135m_industrial_merged"
print("Zipping merged model directory for direct download...")
shutil.make_archive(zip_name, 'zip', OUTPUT_DIR)
print(f"🎉 SUCCESS! You can now download '{zip_name}.zip' from the Output section in the Kaggle sidebar!")